### Section D – Silver reference data

In [0]:
dbutils.widgets.text("catalog", "vp_ce_2026_07_1382", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ---------------------- Silver Stations Clean ---------------------
from pyspark.sql.functions import *

stations_df = spark.read.table(f"{catalog}.bronze.stations")
standardized_stations_df = (
    stations_df
    .drop("_source_file", "_ingested_at")
    .dropDuplicates()
    .withColumns({
        "station_id": expr("NULLIF(UPPER(TRIM(station_id)), '')"),
        "station_name": trim(col("station_name")),
        "city": upper(trim(col("city"))),
        "site_type": upper(trim(col("site_type"))),
        "opened_on": try_to_date(col("opened_on"), "yyyy-MM-dd")
    })
)

standardized_stations_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stations")

In [0]:
# ---------------------- Silver Tariffs Clean ---------------------
from pyspark.sql.functions import *

tariffs_df = spark.read.table(f"{catalog}.bronze.tariffs")
standardized_tariffs_df = (
    tariffs_df
    .drop("_source_file", "_ingested_at")
    .dropDuplicates()
    .withColumns({
        "connector_type": upper(trim(col("connector_type"))),
        "time_band": upper(trim(col("time_band"))),
        "rate_per_kwh": col("rate_per_kwh").cast("decimal(10,2)"),
        "idle_fee_per_min": col("idle_fee_per_min").cast("decimal(10,2)"),
        "free_idle_minutes": col("free_idle_minutes").cast("int")
    })
)

standardized_tariffs_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.tariffs")

In [0]:
# ---------------------- Silver Chargers Clean ---------------------
from pyspark.sql.functions import *
from pyspark.sql.window import Window

chargers_df = spark.read.table(f"{catalog}.bronze.chargers")
standardized_chargers_df = (
    chargers_df
    .withColumns({
        "charger_id": expr("NULLIF(UPPER(TRIM(charger_id)), '')"),
        "station_id": expr("NULLIF(UPPER(TRIM(station_id)), '')"),
        "connector_type": upper(trim(col("connector.type"))),
        "max_kw": col("connector.max_kw").cast("decimal(6,1)"),
        "is_active": col("is_active").cast("boolean"),
        "commissioned_on": try_to_date(col("commissioned_on"), "yyyy-MM-dd"),
        "last_changed_snapshot_date": try_to_date(col("snapshot_date"), "yyyy-MM-dd")
    })
    .drop("_source_file", "_ingested_at", "connector")
    .dropDuplicates()
)

# Keep only the latest snapshot per charger
latest_chargers_df = (
    standardized_chargers_df
    .withColumn("_rn", row_number().over(
        Window.partitionBy("charger_id").orderBy(col("last_changed_snapshot_date").desc())
    ))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

latest_chargers_df.createOrReplaceTempView("staging_chargers")

In [0]:
%sql
-- Silver Chargers – MERGE INTO with CDF

CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.silver.chargers') AS
SELECT * FROM staging_chargers WHERE 1 = 0;

ALTER TABLE IDENTIFIER(:catalog || '.silver.chargers')
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

MERGE INTO IDENTIFIER(:catalog || '.silver.chargers') AS tgt
USING staging_chargers AS src
ON tgt.charger_id = src.charger_id
WHEN MATCHED AND (
     NOT (tgt.station_id    <=> src.station_id)
  OR NOT (tgt.connector_type <=> src.connector_type)
  OR NOT (tgt.max_kw        <=> src.max_kw)
  OR NOT (tgt.is_active     <=> src.is_active)
  OR NOT (tgt.commissioned_on <=> src.commissioned_on)
)
THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
# ---------------------- Silver Members Clean ---------------------
from pyspark.sql.functions import *
from pyspark.sql.window import Window

members_df = spark.read.table(f"{catalog}.bronze.members")

standardized_members_df = (
    members_df
    .drop("_source_file", "_ingested_at")
    .dropDuplicates()
    .withColumns({
        "member_id": expr("NULLIF(UPPER(TRIM(member_id)), '')"),
        "full_name": trim(col("full_name")),
        "email": lower(trim(col("email"))),
        "phone": expr("right(regexp_replace(phone, '[^0-9]', ''), 10)"),
        "city": upper(trim(col("city"))),
        "plan": expr("NULLIF(UPPER(TRIM(plan)), '')"),
        "joined_on": coalesce(
            try_to_date(col("joined_on"), "yyyy-MM-dd"),
            try_to_date(col("joined_on"), "dd/MM/yyyy")
        ),
        "effective_from": coalesce(
            try_to_date(col("change_effective_date"), "yyyy-MM-dd"),
            try_to_date(col("change_effective_date"), "dd/MM/yyyy"),
            try_to_date(col("joined_on"), "yyyy-MM-dd"),
            try_to_date(col("joined_on"), "dd/MM/yyyy")
        )
    })
)

w = Window.partitionBy("member_id").orderBy(col("effective_from").asc())
standardized_members_df = (
    standardized_members_df
    .withColumn("effective_to", lead(col("effective_from")).over(w))
    .withColumn("is_current", col("effective_to").isNull())
)

standardized_members_df = standardized_members_df.select(
    "member_id",
    "full_name",
    "email",
    "phone",
    "city",
    "plan",
    "joined_on",
    "effective_from",
    "effective_to",
    "is_current"
)

standardized_members_df.createOrReplaceTempView("staging_members")

In [0]:
# ---------------------- Silver Members SCD Type 2 ---------------------

# 1. Create the target table if it does not exist (schema-only)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER("{catalog}.silver.members") AS
SELECT * FROM staging_members WHERE 1 = 0
""")

# 2. MERGE using only the latest staged row per member (is_current = true).
spark.sql(f"""
MERGE INTO {catalog}.silver.members AS t
USING (SELECT * FROM staging_members WHERE is_current = true) AS s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND (
       NOT (t.plan <=> s.plan)
    OR NOT (t.city  <=> s.city)
)
THEN UPDATE SET
    t.effective_to = s.effective_from,
    t.is_current   = false
WHEN MATCHED AND (
       NOT (t.full_name <=> s.full_name)
    OR NOT (t.email     <=> s.email)
    OR NOT (t.phone     <=> s.phone)
)
THEN UPDATE SET
    t.full_name = s.full_name,
    t.email     = s.email,
    t.phone     = s.phone
WHEN NOT MATCHED THEN INSERT
    (member_id, full_name, email, phone, city, plan, joined_on,
     effective_from, effective_to, is_current)
    VALUES
    (s.member_id, s.full_name, s.email, s.phone, s.city, s.plan, s.joined_on,
     s.effective_from, NULL, true)
""")

# 3. Insert new SCD2 versions for members whose current version was just closed.
#    The NOT EXISTS guard makes this step idempotent on re-processing.
spark.sql(f"""
INSERT INTO {catalog}.silver.members
SELECT
    s.member_id,
    s.full_name,
    s.email,
    s.phone,
    s.city,
    s.plan,
    s.joined_on,
    s.effective_from,
    NULL AS effective_to,
    true AS is_current
FROM (SELECT * FROM staging_members WHERE is_current = true) s
WHERE EXISTS (
    SELECT 1 FROM {catalog}.silver.members t
    WHERE t.member_id = s.member_id
      AND t.effective_to = s.effective_from
      AND t.is_current = false
)
AND NOT EXISTS (
    SELECT 1 FROM {catalog}.silver.members t
    WHERE t.member_id = s.member_id
      AND t.effective_from = s.effective_from
      AND t.is_current = true
)
""")

### Section C – Silver sessions and data quality

In [0]:
# Silver Sessions – Standardise, De-duplicate, Validate, Output
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sessions_df = spark.read.table(f"{catalog}.bronze.sessions")
raw_cols = sessions_df.columns

# --- C1: Standadize ---
standardized_df = (
    sessions_df
    .dropDuplicates()
    .withColumn("raw_record", to_json(struct(*[col(c) for c in raw_cols])))
    .withColumns({
        "session_id":  expr("NULLIF(UPPER(TRIM(session_id)), '')"),
        "charger_id": expr("NULLIF(UPPER(TRIM(charger_id)), '')"),
        "member_id":   expr("NULLIF(UPPER(TRIM(member_id)), '')"),
        "status": upper(trim("status")),
        "started_at": coalesce(
            try_to_timestamp(col("started_at"), lit("yyyy-MM-dd HH:mm:ss")),
            try_to_timestamp(col("started_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
        ),
        "ended_at": coalesce(
            try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd HH:mm:ss")),
            try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
        )
    })
    .withColumn("session_date", col("started_at").cast("date"))

    .withColumn("energy_kwh",
        when(col("meter.end_kwh").isNotNull() & col("meter.start_kwh").isNotNull(),
             (col("meter.end_kwh") - col("meter.start_kwh")).cast("decimal(10,3)"))
        .otherwise(lit(None).cast("decimal(10,3)"))
    )

    .withColumn("duration_minutes",
        when(col("ended_at").isNotNull() & col("started_at").isNotNull(),
             floor((col("ended_at").cast("long") - col("started_at").cast("long")) / 60).cast("int"))
        .otherwise(lit(None).cast("int"))
    )

    .withColumn("time_band",
        when(col("started_at").isNotNull() & hour(col("started_at")).between(18, 21), lit("PEAK"))
        .when(col("started_at").isNotNull(), lit("OFF_PEAK"))
        .otherwise(lit(None).cast("string"))
    )

    .withColumn("idle_minutes", coalesce(col("idle_minutes").cast("int"), lit(0)))
)

#  C2: De-duplicate (keep latest updated_at per session_id) 
w_dedup = Window.partitionBy("session_id").orderBy(col("updated_at").desc())
deduped_df = (
    standardized_df
    .withColumn("_rn", row_number().over(w_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- C3: Validate and quarantine ---
# Referential integrity: join with silver.chargers and silver.members
chargers_ref = (
    spark.read.table(f"{catalog}.silver.chargers")
    .select("charger_id")
    .withColumn("_in_chargers", lit(True))
)
members_ref = (
    spark.read.table(f"{catalog}.silver.members")
    .select("member_id")
    .withColumn("_in_members", lit(True))
)

enriched_df = (
    deduped_df
    .join(chargers_ref, "charger_id", "left")
    .join(members_ref, "member_id", "left")
)

validated_df = (
    enriched_df
    .withColumn(
        "reject_reason",
        when(col("session_id").isNull(), lit("session_id is required"))
        .when(col("charger_id").isNull() | col("_in_chargers").isNull(),
              lit("R1 UNKNOWN_CHARGER"))
        .when(col("member_id").isNotNull() & col("_in_members").isNull(),
              lit("R2 UNKNOWN_MEMBER"))
        .when(col("started_at").isNull(), lit("R3 INVALID_TIMESTAMP"))
        .when(
            col("status").isin("COMPLETED", "FAILED")
            & (col("ended_at").isNull() | (col("ended_at") <= col("started_at"))),
            lit("R4 INVALID_DURATION")
        )
        .when(
            (col("status") == "COMPLETED")
            & (col("energy_kwh").isNull() | (col("energy_kwh") < 0)),
            lit("R5 INVALID_METER")
        )
        .otherwise(lit(None).cast("string"))
    )
    .drop("_in_chargers", "_in_members")
)

valid_df      = validated_df.filter(col("reject_reason").isNull())
quarantine_df = validated_df.filter(col("reject_reason").isNotNull())

# --- C4: Output tables (incremental MERGE) ---

# silver.sessions
silver_sessions_df = valid_df.select(
    col("session_id"),
    col("charger_id"),
    col("member_id"),
    col("started_at").alias("started_at"),
    col("ended_at").alias("ended_at"),
    col("session_date"),
    col("status"),
    col("time_band"),
    col("energy_kwh"),
    col("duration_minutes"),
    col("idle_minutes"),
    col("payment.amount_inr").cast("decimal(10,2)").alias("amount_paid"),
    col("updated_at")
)

silver_sessions_df.createOrReplaceTempView("staging_sessions")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.sessions AS
SELECT * FROM staging_sessions WHERE 1 = 0
""")

spark.sql(f"""
MERGE INTO {catalog}.silver.sessions AS tgt
USING staging_sessions        AS src
ON tgt.session_id = src.session_id
WHEN MATCHED AND (
     NOT (tgt.charger_id        <=> src.charger_id)
  OR NOT (tgt.member_id         <=> src.member_id)
  OR NOT (tgt.started_at        <=> src.started_at)
  OR NOT (tgt.ended_at          <=> src.ended_at)
  OR NOT (tgt.session_date      <=> src.session_date)
  OR NOT (tgt.status            <=> src.status)
  OR NOT (tgt.time_band         <=> src.time_band)
  OR NOT (tgt.energy_kwh        <=> src.energy_kwh)
  OR NOT (tgt.duration_minutes  <=> src.duration_minutes)
  OR NOT (tgt.idle_minutes      <=> src.idle_minutes)
  OR NOT (tgt.amount_paid       <=> src.amount_paid)
  OR NOT (tgt.updated_at        <=> src.updated_at)
)
THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# silver.session_faults — one row per fault of a valid session (S7: trim, upper fault_code)
silver_session_faults_df = (
    valid_df
    .select(col("session_id"), posexplode(col("faults")))
    .toDF("session_id", "pos", "faults")
    .select(
        col("session_id"),
        (col("pos") + 1).alias("fault_seq"),
        upper(trim(col("faults.code"))).alias("fault_code")
    )
)

silver_session_faults_df.createOrReplaceTempView("staging_session_faults")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.session_faults AS
SELECT * FROM staging_session_faults WHERE 1 = 0
""")

spark.sql(f"""
MERGE INTO {catalog}.silver.session_faults AS tgt
USING staging_session_faults        AS src
ON tgt.session_id = src.session_id AND tgt.fault_seq = src.fault_seq
WHEN MATCHED AND (
     NOT (tgt.fault_code <=> src.fault_code)
)
THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# silver.quarantine_sessions — one row per rejected session
silver_quarantine_df = quarantine_df.select(
    col("session_id"),
    col("reject_reason"),
    col("updated_at"),
    col("raw_record"),
    current_timestamp().alias("quarantined_at")
)

silver_quarantine_df.createOrReplaceTempView("staging_quarantine_sessions")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_sessions AS
SELECT * FROM staging_quarantine_sessions WHERE 1 = 0
""")

spark.sql(f"""
MERGE INTO {catalog}.silver.quarantine_sessions AS tgt
USING staging_quarantine_sessions        AS src
ON tgt.session_id = src.session_id
WHEN MATCHED AND (
     NOT (tgt.reject_reason <=> src.reject_reason)
  OR NOT (tgt.updated_at    <=> src.updated_at)
  OR NOT (tgt.raw_record     <=> src.raw_record)
)
THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")